In [1]:
import pandas as pd
from pathlib import Path

In [2]:
#Entrando no caminho dos arquivos
DATA_PATH = Path("../data/raw")

files = list(DATA_PATH.glob("*.csv"))

for file in files:
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [3]:
# Carregando os datasets em um dicionário
# CEP é lido como string: como inteiro os zeros à esquerda se perdem (ex.: 01046 -> 1046)
zip_dtypes = {
    "customer_zip_code_prefix": str,
    "seller_zip_code_prefix": str,
    "geolocation_zip_code_prefix": str,
}

datasets = {}

for file in files:
    name = file.stem
    datasets[name] = pd.read_csv(file, dtype=zip_dtypes)

print(f"{len(datasets)} datasets carregados.")

9 datasets carregados.


In [4]:
# Printando a forma de cada dataset
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

olist_customers_dataset: (99441, 5)
olist_geolocation_dataset: (1000163, 5)
olist_orders_dataset: (99441, 8)
olist_order_items_dataset: (112650, 7)
olist_order_payments_dataset: (103886, 5)
olist_order_reviews_dataset: (99224, 7)
olist_products_dataset: (32951, 9)
olist_sellers_dataset: (3095, 4)
product_category_name_translation: (71, 2)


In [5]:
# Carregando dataframes
customers_df = pd.DataFrame(datasets["olist_customers_dataset"])
orders_df = pd.DataFrame(datasets["olist_orders_dataset"])
geolocation_df = pd.DataFrame(datasets["olist_geolocation_dataset"])
customers_df.shape

(99441, 5)

In [6]:
customers_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  str  
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: str(5)
memory usage: 3.8 MB


In [7]:
#Avaliando se existe valores nulos
customers_df.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [8]:
#Avaliando se existe dados duplicados
print(f"Linhas totalmente duplicadas: {customers_df.duplicated().sum()}")
print(f"customer_id duplicados: {customers_df['customer_id'].duplicated().sum()}")

Linhas totalmente duplicadas: 0
customer_id duplicados: 0


In [9]:
# customer_id é gerado a cada pedido; customer_unique_id identifica a pessoa
unique_people = customers_df['customer_unique_id'].nunique()
ids_per_person = customers_df.groupby('customer_unique_id')['customer_id'].nunique()

print(f"customer_id distintos: {customers_df['customer_id'].nunique()}")
print(f"Clientes únicos (customer_unique_id): {unique_people}")
print(f"Clientes com mais de um customer_id (recompra): {(ids_per_person > 1).sum()} ({(ids_per_person > 1).mean() * 100:.2f}%)")

customer_id distintos: 99441
Clientes únicos (customer_unique_id): 96096
Clientes com mais de um customer_id (recompra): 2997 (3.12%)


In [10]:
# CEP: precisa ter 5 dígitos (string preserva os zeros à esquerda)
print(customers_df['customer_zip_code_prefix'].dtype)
print(customers_df['customer_zip_code_prefix'].str.len().value_counts())
print(f"CEPs iniciados em zero: {customers_df['customer_zip_code_prefix'].str.startswith('0').sum()}")

str
customer_zip_code_prefix
5    99441
Name: count, dtype: int64
CEPs iniciados em zero: 23995


In [11]:
UFS = {"AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG", "PA",
       "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"}

# Estados: devem ser UFs válidas
invalid_states = customers_df.loc[~customers_df['customer_state'].isin(UFS), 'customer_state'].unique()
print(f"Estados distintos: {customers_df['customer_state'].nunique()}")
print(f"Estados inválidos: {invalid_states}")
customers_df['customer_state'].value_counts().head(10)

Estados distintos: 27
Estados inválidos: <StringArray>
[]
Length: 0, dtype: str


customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64

In [12]:
# Cidades: espaços, maiúsculas e caracteres suspeitos
cities = customers_df['customer_city']

print(f"Cidades distintas: {cities.nunique()}")
print(f"Com espaços nas pontas ou maiúsculas: {(cities != cities.str.strip().str.lower()).sum()}")
print(f"Com dígitos ou símbolos suspeitos: {cities.str.contains(r'[0-9@\\/]').sum()}")
cities[cities.str.contains(r"[0-9@\\/]")]

Cidades distintas: 4119
Com espaços nas pontas ou maiúsculas: 0
Com dígitos ou símbolos suspeitos: 1


6333    quilometro 14 do mutum
Name: customer_city, dtype: str

In [13]:
# Integridade com orders
print(f"customer_id sem pedido em orders: {(~customers_df['customer_id'].isin(orders_df['customer_id'])).sum()}")
print(f"Pedidos cujo customer_id não existe em customers: {(~orders_df['customer_id'].isin(customers_df['customer_id'])).sum()}")

customer_id sem pedido em orders: 0


Pedidos cujo customer_id não existe em customers: 0


In [14]:
# Cobertura do CEP na tabela de geolocalização
missing_zip = ~customers_df['customer_zip_code_prefix'].isin(geolocation_df['geolocation_zip_code_prefix'])
print(f"Clientes com CEP sem correspondência na geolocation: {missing_zip.sum()} ({missing_zip.mean() * 100:.2f}%)")

Clientes com CEP sem correspondência na geolocation: 278 (0.28%)

In [15]:
#Fazendo copia do dataframe para não alterar o original
customers_clean = customers_df.copy()

In [16]:
# Padronização de texto e CEP (sem remover registros)
customers_clean['customer_city'] = customers_clean['customer_city'].str.strip().str.lower()
customers_clean['customer_state'] = customers_clean['customer_state'].str.strip().str.upper()
customers_clean['customer_zip_code_prefix'] = customers_clean['customer_zip_code_prefix'].str.strip().str.zfill(5)

In [17]:
# Verificando quantos valores foram alterados em cada coluna
for col in customers_df.columns:
    print(f"{col}: {(customers_df[col] != customers_clean[col]).sum()} valores alterados")

customer_id: 0 valores alterados
customer_unique_id: 0 valores alterados
customer_zip_code_prefix: 0 valores alterados
customer_city: 0 valores alterados
customer_state: 0 valores alterados


In [18]:
# Validações pós-tratamento
print(f"Linhas: {customers_clean.shape[0]} (original: {customers_df.shape[0]})")
print(f"Valores nulos: {customers_clean.isna().sum().sum()}")
print(f"customer_id duplicados: {customers_clean['customer_id'].duplicated().sum()}")
print(f"CEPs com tamanho diferente de 5: {(customers_clean['customer_zip_code_prefix'].str.len() != 5).sum()}")
print(f"Estados inválidos: {(~customers_clean['customer_state'].isin(UFS)).sum()}")
print(f"customer_id sem pedido em orders: {(~customers_clean['customer_id'].isin(orders_df['customer_id'])).sum()}")

assert customers_clean.shape == customers_df.shape
assert customers_clean['customer_id'].is_unique
assert customers_clean['customer_zip_code_prefix'].str.len().eq(5).all()

Linhas: 99441 (original: 99441)
Valores nulos: 0
customer_id duplicados: 0
CEPs com tamanho diferente de 5: 0
Estados inválidos: 0


customer_id sem pedido em orders: 0


In [19]:
# Salvando dataframe tratado em pasta processed
customers_clean.to_csv('../data/processed/olist_customers_dataset_clean.csv', index=False)

## 📌 Conclusão da etapa — Customers

### 🔎 O que foi encontrado?

A tabela `customers` possui **99.441 registros e 5 colunas**.

Durante o profiling, foram identificados:

- Nenhum valor nulo e nenhuma duplicidade (`customer_id` é único);
- **96.096 clientes únicos** (`customer_unique_id`): apenas **2.997 (3,12%)** possuem mais de um `customer_id`, ou seja, recompraram;
- 27 estados, todos UFs válidas, e todos os CEPs com 5 dígitos;
- **23.995 CEPs começam com zero**. Lendo o CSV com o tipo padrão do pandas, o CEP vira inteiro e perde o zero (`01046` → `1046`);
- 1 cidade com número no nome (`quilometro 14 do mutum`), que é um nome de localidade real;
- **278 clientes (0,28%)** com CEP que não existe em `geolocation`.

### 🛠️ O que foi feito?

Os dados já estavam padronizados (cidade em minúsculo e sem acento, UF em maiúsculo). Foram aplicadas apenas as padronizações preventivas (`strip`, `lower`, `upper` e `zfill(5)` no CEP) e **nenhum valor foi alterado**. Nenhum registro foi removido.

O ponto principal desta etapa é que o CEP é carregado e salvo como **string**, preservando os zeros à esquerda.

### ✅ Como foi validado?

- A quantidade de registros permaneceu em **99.441**;
- Sem nulos e sem `customer_id` duplicado;
- Todos os CEPs com 5 dígitos e todos os estados válidos;
- Todos os `customer_id` possuem pedido em `orders` e todo pedido possui cliente.

### 📁 Resultado

O dataset tratado foi salvo em:

`data/processed/olist_customers_dataset_clean.csv`

> ⚠️ Ao ler este arquivo em outras etapas, usar `dtype={"customer_zip_code_prefix": str}`.